In [1]:
import pandas as pd
import numpy as np


In [2]:
# check mdna 

mdna = pd.read_csv('./data/mdna_facts_top_tech_test.csv')
print(mdna.head(1))

print(mdna.info())

      cik ticker      accession_number form_type filing_date report_date  \
0  320193   AAPL  0000320193-23-000106      10-K  2023-11-03  2023-09-30   

  section      fact_type direction  \
0    MD&A  MARGIN_DRIVER   neutral   

                                       evidence_text  \
0  Products gross margin percentage increased dur...   

                                          source_url  sent_index  
0  https://www.sec.gov/Archives/edgar/data/320193...          32  
<class 'pandas.DataFrame'>
RangeIndex: 1034 entries, 0 to 1033
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   cik               1034 non-null   int64
 1   ticker            1034 non-null   str  
 2   accession_number  1034 non-null   str  
 3   form_type         1034 non-null   str  
 4   filing_date       1034 non-null   str  
 5   report_date       1034 non-null   str  
 6   section           1034 non-null   str  
 7   fact_type        

In [3]:
# check filtered 

filtered = pd.read_csv('./data/filtered_tickers.csv')
print(filtered.head(1))

print(filtered.info())

         cik ticker  accession_number     form_type filing_date       section  \
0  1318605.0   TSLA               NaN  news_article         NaN  news_article   

      fact_type direction                                      evidence_text  \
0  news_article  positive  The fatal Dec. 29 crash of aTeslavehicle in So...   

                                          source_url  ...  language  \
0  https://www.cnbc.com/2019/12/31/us-auto-safety...  ...   English   

   sourcecountry                                               text  \
0  United States  The fatal Dec. 29 crash of aTeslavehicle in So...   

      source_file news_site title_date text_date headline_direction  \
0  2020-01-01.csv  cnbc.com        NaN       NaN            neutral   

  text_direction company_name  
0       positive  Tesla, Inc.  

[1 rows x 27 columns]
<class 'pandas.DataFrame'>
RangeIndex: 9857 entries, 0 to 9856
Data columns (total 27 columns):
 #   Column              Non-Null Count  Dtype  
---  ------    

In [4]:
# normalize tickers 
filtered["ticker"] = filtered["ticker"].str.upper().str.strip()
mdna["ticker"] = mdna["ticker"].str.upper().str.strip()

# Build ticker → company_name map from filtered
ticker_map = (
    filtered[["ticker", "company_name"]]
    .drop_duplicates()
    .set_index("ticker")["company_name"]
)

# Map company names into mdna
mdna["company_name"] = mdna["ticker"].map(ticker_map)

# create desired columns 

cols = [
    "ticker",
    "company_name",
    "evidence_text",
    "form_type",
    "filing_date",
    ""
    "source_url",
]

df_filtered = filtered[cols]
df_mdna = mdna[cols]

In [5]:
# check if any missing fields 

print(df_mdna.isna().sum())
print(df_filtered.isna().sum())

ticker            0
company_name     75
evidence_text     0
form_type         0
filing_date       0
source_url        0
dtype: int64
ticker              3
company_name        3
evidence_text      11
form_type           0
filing_date      6742
source_url          0
dtype: int64


In [6]:
# check missing MDNA company names

df_mdna[df_mdna['company_name'].isna()]['ticker'].value_counts()

ticker
QCOM    75
Name: count, dtype: int64

In [7]:
# add Qualcomm Inc as company name 

df_mdna.loc[
    df_mdna['company_name'].isna(), 
    'company name'] = 'Qualcomm Inc'

In [8]:
# check missing ticker/company name in Filtered

df_filtered[df_filtered['ticker'].isna()]

,ticker,company_name,evidence_text,form_type,filing_date,source_url
1967,NaN,NaN,Amazon is suspending its distribution activity...,news_article,NaN,https://www.cnn.com/2020/04/15/tech/amazon-fra...
2060,NaN,NaN,Amazon has removed merchandise sold by third p...,news_article,NaN,https://www.cnn.com/2020/08/19/tech/amazon-rem...
3300,NaN,NaN,(CNN) -- تعزز السعودية علاقاتها مع شركات الذكا...,news_article,NaN,https://arabic.cnn.com/middle-east/article/202...


In [ ]:
# remove above rows since no way to reconcile missing ticker
# AND missing company 

df_filtered = df_filtered.drop([1967, 2060, 3300])

In [9]:
# remove rows with missing evidence text 

df_filtered = df_filtered[~df_filtered['evidence_text'].isna()]

df_filtered.isna().sum()

ticker              3
company_name        3
evidence_text       0
form_type           0
filing_date      6731
source_url          0
dtype: int64

In [10]:
# finally, we concatenate into a single df 

df_all = pd.concat([df_filtered, df_mdna], ignore_index=True)

df_all.head(1)

,ticker,company_name,evidence_text,form_type,filing_date,source_url,company name
0,TSLA,"Tesla, Inc.",The fatal Dec. 29 crash of aTeslavehicle in So...,news_article,NaN,https://www.cnbc.com/2019/12/31/us-auto-safety...,NaN


In [11]:
# remove any duplicates 
df_all = df_all.drop_duplicates(subset=["ticker", "evidence_text"])
df_all.head(1)

,ticker,company_name,evidence_text,form_type,filing_date,source_url,company name
0,TSLA,"Tesla, Inc.",The fatal Dec. 29 crash of aTeslavehicle in So...,news_article,NaN,https://www.cnbc.com/2019/12/31/us-auto-safety...,NaN


In [12]:
# send into new .csv 

df_all.to_csv('./data/combined.csv', index=False)